In [14]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
!pip install langchain-google-vertexai

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.4 MB 7.4 MB/s eta 0:00:01
   ------------ --------------------------- 2.6/8.4 MB 6.6 MB/s eta 0:00:01
   ------------------ --------------------- 3.9/8.4 MB 6.3 MB/s eta 0:00:01
   ----------------------- ---------------- 5.0/8.4 MB 6.0 MB/s eta 0:00:01
   ---------------------------- ----------- 6.0/8.4 MB 5.5 MB/s eta 0:00:01
   -------------------------------- ------- 6.8/8.4 MB 5.4 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.4 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 4.9 MB/s  0:00:01
   ---------------------------------------- 0.0/28.0 MB ? eta -:--:--
   - -------------------------------------- 0.8/28.0 MB 3.7 MB/s eta 0:00:08
   -- ------------------------------------- 1.6/28.0 MB 3.8 MB/s eta 0:00:07
   --- ------------------------------------ 2.4/28.0 MB 3.5 MB/s eta 0:00:08
   ---- -----------------

In [5]:
!pip install -U langchain-google-genai

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI


In [15]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable


large_model = init_chat_model("gpt-4o-mini")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model


    request = request.override(model=model)  

    return handler(request)

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [17]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Short answer: I didn’t water it today. I’m not there to water it physically.

I can help with:
- Setting a daily or every-other-day reminder to water it.
- Logging the last watering time for your plant care checklist.

If you tell me the plant type and its usual schedule, I can tailor the reminder. Quick tip: many office plants dry out a bit between waterings—check the soil about an inch down before watering. Would you like me to set up a reminder?


In [18]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [19]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

We should consider repotting when the roots start coming out of the drainage holes, usually in about a year or so.


In [20]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-4o-mini-2024-07-18
